In [ ]:
import pandas as pd
import json

# ==== ĐỌC FILE ====
with open("config_single.json", "r", encoding="utf-8") as file:
    config = json.load(file)

# OUTPUT_CSV = config.get("OUTPUT_CSV")
OUTPUT_CSV = "loc_single.csv"
print("OUTPUT_CSV : ", OUTPUT_CSV)
df = pd.read_csv(OUTPUT_CSV)
print(f"Số dòng ban đầu: {len(df)}")

# ==== ÉP KIỂU SỐ ====
numeric_cols = ["sharpe", "fitness", "LOW_2Y_SHARPE_value"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ==== LỌC ====
filtered_df = df[
    (
        (df["sharpe"] >= 0.8) &
        (
            (df["LOW_2Y_SHARPE_value"] > 0) |
            (df["LOW_2Y_SHARPE_value"].isna())
        )
    )
    |
    (
        (df["sharpe"] <= -0.8) &
        (
            (df["LOW_2Y_SHARPE_value"] < 0) |
            (df["LOW_2Y_SHARPE_value"].isna())
        )
    )
].copy()

# ==== FLIP ALPHA ÂM ====
mask_negative = filtered_df["sharpe"] <= -0.8

filtered_df.loc[mask_negative, ["sharpe", "fitness", "LOW_2Y_SHARPE_value"]] = (
    filtered_df.loc[mask_negative, ["sharpe", "fitness", "LOW_2Y_SHARPE_value"]].abs()
)

filtered_df.loc[mask_negative, "regular"] = (
    "-" + filtered_df.loc[mask_negative, "regular"].astype(str)
)

# ==== CHỌN CỘT ====
output_df = filtered_df[
    ["id", "sharpe", "fitness", "LOW_2Y_SHARPE_value", "regular"]
]

# ==== SORT (QUAN TRỌNG: để giữ bản tốt nhất) ====
output_df = output_df.sort_values(by="sharpe", ascending=False)

# ==== REMOVE DUPLICATE THEO REGULAR ====
output_df = output_df.drop_duplicates(subset=["regular"], keep="first")

# ==== RESET INDEX ====
output_df = output_df.reset_index(drop=True)

# ==== LƯU ====
output_df.to_csv("loc_single.csv", index=False)

print(f"Done ✅ Số alpha sau lọc & dedup: {len(output_df)}")

OUTPUT_CSV :  loc_single.csv
Số dòng ban đầu: 4125
Done ✅ Số alpha sau lọc & dedup: 2645


In [1]:
import pandas as pd
import requests
import re
import os
import json

import os

CURRENT_DIR = os.getcwd()
BASE_DIR = os.path.dirname(CURRENT_DIR)
SESSION_FILE = os.path.join(BASE_DIR, "session.json")
LOC_FILE = "loc_single.csv"
OPERATOR_FILE = "operator_generator.csv"

def load_session():
    if not os.path.exists(SESSION_FILE):
        print("❌ Không tìm thấy session.json")
        return None
    try:
        with open(SESSION_FILE, "r", encoding="utf-8") as f: token = json.load(f).get("t")
        if not token: print("❌ session.json không có token t")
        return token
    except Exception as e:
        print(f"❌ Lỗi load session: {e}")
        return None

def load_headers():
    token = load_session()
    return {"Content-Type": "application/json", "Cookie": f"t={token}"} if token else {}

def get_functions(expression):
    return set(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\(', str(expression)))

def load_operator_file():
    paths = [
        os.path.join(CURRENT_DIR, OPERATOR_FILE),
        os.path.join(BASE_DIR, OPERATOR_FILE)
    ]

    for path in paths:
        if os.path.exists(path):
            try:
                df_op = pd.read_csv(path)
                if "name" not in df_op.columns:
                    print(f"⚠️ File không có cột name: {path}")
                    continue

                valid_operators = set(df_op["name"].dropna().astype(str).str.strip())
                valid_operators.discard("")

                if valid_operators:
                    print(f"📂 Dùng operator từ file: {path}")
                    print(f"✅ Operator trong file: {len(valid_operators)}")
                    return valid_operators
            except Exception as e:
                print(f"⚠️ Lỗi đọc {path}: {e}")

    raise SystemExit(
        f"❌ Không tìm thấy {OPERATOR_FILE} hợp lệ ở:\n"
        f"   {CURRENT_DIR}\n"
        f"   {BASE_DIR}"
    )

headers = load_headers()
valid_operators = None

if headers:
    try:
        r = requests.get(
            "https://api.worldquantbrain.com/operators",
            headers=headers,
            timeout=60
        )

        if r.status_code == 200:
            data = r.json()
            operators = data if isinstance(data, list) else data.get("results", [])

            if operators:
                pd.DataFrame(operators).to_csv(
                    OPERATOR_FILE,
                    index=False,
                    encoding="utf-8-sig"
                )

                valid_operators = {
                    str(x.get("name")).strip()
                    for x in operators
                    if isinstance(x, dict) and x.get("name")
                }

                print(f"✅ Operator WQ hiện có: {len(valid_operators)}")
                print(f"✅ Đã lưu vào {OPERATOR_FILE}")

            else:
                print("⚠️ WQ không trả về operator → dùng file operator_single.csv")

        elif r.status_code == 401:
            print("🔐 Session đã hết hạn → dùng file operator_single.csv")

        else:
            print(
                f"⚠️ GET operators lỗi: {r.status_code} | "
                f"{r.text[:500]} → dùng file operator_single.csv"
            )

    except requests.exceptions.Timeout:
        print("⚠️ GET operators timeout → dùng file operator_single.csv")

    except requests.exceptions.RequestException as e:
        print(f"⚠️ Lỗi GET operators: {e} → dùng file operator_single.csv")

else:
    print("⚠️ Không load được session → dùng file operator_single.csv")

# Nếu API không lấy được operator thì tìm file local
if not valid_operators:
    valid_operators = load_operator_file()

df = pd.read_csv(LOC_FILE)
if "regular" not in df.columns: raise SystemExit("❌ loc_single.csv không có cột regular")

old_count = len(df)

df["_functions"] = df["regular"].fillna("").apply(get_functions)
df["_invalid"] = df["_functions"].apply(lambda x: x - valid_operators)

all_invalid = sorted(set().union(*df["_invalid"]) if len(df) else set())

# print("\n===== HÀM KHÔNG ĐƯỢC DÙNG =====")
# if not all_invalid:
#     print("✅ Không phát hiện hàm không hợp lệ")
# else:
#     for op in all_invalid:
#         count = df["_invalid"].apply(lambda x: op in x).sum()
#         print(f"{op}: {count} hàng")

invalid_mask = df["_invalid"].apply(bool)
removed_count = int(invalid_mask.sum())

df = df[~invalid_mask].drop(columns=["_functions", "_invalid"])
new_count = len(df)

df.to_csv(LOC_FILE, index=False, encoding="utf-8-sig")

print("\n===== KẾT QUẢ =====")
print(f"Số hàng cũ: {old_count}")
print(f"Số hàng bị loại: {removed_count}")
print(f"Số hàng sau khi lọc: {new_count}")
print(f"✅ Đã ghi lại file: {LOC_FILE}")

✅ Operator WQ hiện có: 104
✅ Đã lưu vào operator_generator.csv

===== KẾT QUẢ =====
Số hàng cũ: 1307
Số hàng bị loại: 557
Số hàng sau khi lọc: 750
✅ Đã ghi lại file: loc_single.csv


In [3]:
import pandas as pd
import random
import os
# Tham số
n = 6  # <-- Bạn có thể thay đổi giá trị này
num_formulas = 50000

# Đọc dữ liệu
df = pd.read_csv("loc_single.csv")

# Chia theo sharpe
regular_ge_1 = df[df["sharpe"] >= 0.8]["regular"].tolist()
regular_lt_1 = df[df["sharpe"] < 0.8]["regular"].tolist()

def generate_combination(n):
    p = random.random()
    
    # Chọn theo xác suất
    # if p <= 1:
    selected = random.sample(regular_ge_1, min(n, len(regular_ge_1)))
    # elif p < 0.36:
    #     selected = random.sample(regular_lt_1, min(n, len(regular_lt_1)))
    # else:
    #     max_ge = min(len(regular_ge_1), n - 2)
    #     count_ge = random.randint(2, max_ge) if max_ge >= 2 else max_ge
    #     count_lt = n - count_ge
    #     if count_ge > 0 and count_lt > 0:
    #         selected = random.sample(regular_ge_1, count_ge) + \
    #                    random.sample(regular_lt_1, min(count_lt, len(regular_lt_1)))
    #     else:
    #         selected = random.sample(regular_ge_1 + regular_lt_1, n)

    # Shuffle để đảm bảo thứ tự ngẫu nhiên
    random.shuffle(selected)

    # # Gán dấu trừ ngẫu nhiên (nếu không phải xác suất 0.25)
    # if random.random() >= 0.95:
    #     count_minus = random.randint(1, min(2, len(selected)))
    #     minus_indices = random.sample(range(len(selected)), count_minus)
    #     # print("minus_indices : ", minus_indices)
        
    #     for i in minus_indices:
    #         expr = selected[i].lstrip()
    #         if expr.startswith("-"):
    #             # Nếu đã có dấu '-', thêm nữa thành '--' → chuyển thành không có dấu
    #             selected[i] = expr.lstrip("-+ ")
    #         else:
    #             # Nếu chưa có '-', thêm vào
    #             selected[i] = "-" + expr.lstrip("+- ")
    #         # print("selected[i] : ", selected[i])

    # Ghép công thức với logic đúng dấu
    expression = selected[0].strip()
    for expr in selected[1:]:
        expr = expr.strip()
        if expr.startswith("-"):
            # Nếu biểu thức đã có dấu trừ → + -expr
            expression += " + " + expr
        elif expr.startswith("+"):
            expression += " + " + expr[1:]
        else:
            expression += " + " + expr

    # Thay thế các chuỗi '+ -' bằng '-'
    expression = expression.replace("+ -", "- ")

    return expression


# Tạo 10,000 công thức (cho phép trùng lặp nhẹ)
formulas = []
for _ in range(num_formulas):
    n = random.choices([6, 7, 8, 9], weights=[0.25, 0.25, 0.25, 0.25])[0]
    formula = generate_combination(n)
    formulas.append(formula)

# Ghi ra file
with open("alpha_single.txt", "w") as f:
    for formula in formulas:
        f.write(formula + "\n")

print(f"✅ Đã tạo {num_formulas} công thức với {n} regular mỗi công thức.")


✅ Đã tạo 50000 công thức với 7 regular mỗi công thức.


In [ ]:
import pandas as pd

FILE = "loc_single.csv"

df = pd.read_csv(FILE)
before = len(df)

df = df[~df["regular"].str.contains("fnd", case=False, na=False)]
df.to_csv(FILE, index=False, encoding="utf-8-sig")

print(f"Đã xóa {before - len(df)} hàng chứa fnd")
print(f"Còn lại {len(df)} hàng")

In [ ]:
import pandas as pd
import re

INPUT_FILE = "loc_single_fit2.csv"
OUTPUT_FILE = "loc_single_norsk.csv"

ALL_VARIABLES = [
    "mdl216_armcountry100score", "mdl216_armestimaterevisioncombinationrank",
    "mdl216_armglobal100score", "mdl216_armglobal1rank",
    "mdl216_armglobalcombinationscore", "mdl216_armpreferredrevision1rank",
    "mdl216_armpreferredrevisionscore", "mdl216_armregion1rank",
    "mdl216_armregion5score", "mdl216_preferredblendedrevisionfy1percent",
    "mdl242_1mt", "mdl242_factor_momentum_component",
    "mdl242_reversal_component", "mdl242_seasonality_component",
    "rsk59_bid_rate", "rsk59_crowded_score",
    "rsk59_daystocover10day", "rsk59_daystocover30day",
    "rsk59_daystocover90day", "rsk59_dtcdailychg",
    "rsk59_indicativeavailability", "rsk59_offer_rate",
    "rsk59_short_interest", "rsk59_squeeze_risk",
    "shrt3_bar",
    "adv20", "cap", "close", "high", "low", "open",
    "returns", "sharesout", "sudden_volume_spike_score",
    "transaction_count", "volume", "vwap",
    "available_market_value_usd", "average_loan_duration_days",
    "loan_rate_volatility", "loan_utilization_ratio",
    "loaned_market_value_usd", "loaned_share_count",
    "max_loan_rate", "mean_loan_rate"
]

VECTOR_VARIABLES = [
    # model216
    "mdl216_armcountry100score",
    "mdl216_armestimaterevisioncombinationrank",
    "mdl216_armglobal100score",
    "mdl216_armglobal1rank",
    "mdl216_armglobalcombinationscore",
    "mdl216_armpreferredrevision1rank",
    "mdl216_armpreferredrevisionscore",
    "mdl216_armregion1rank",
    "mdl216_armregion5score",
    "mdl216_preferredblendedrevisionfy1percent"
]

MATRIX_VARIABLES = [
    # model242
    "mdl242_1mt",
    "mdl242_factor_momentum_component",
    "mdl242_reversal_component",
    "mdl242_seasonality_component",
    "sudden_volume_spike_score",

    # Price-volume
    "adv20",
    "cap",
    "close",
    "high",
    "low",
    "open",
    "returns",
    "sharesout",
    "volume",
    "vwap"
]

ALLOWED_VARIABLES = set(VECTOR_VARIABLES + MATRIX_VARIABLES)
REMOVE_VARIABLES = sorted(set(ALL_VARIABLES) - ALLOWED_VARIABLES)

df = pd.read_csv(INPUT_FILE)
pattern = r'\b(?:' + '|'.join(map(re.escape, REMOVE_VARIABLES)) + r')\b'

mask = ~df["regular"].fillna("").astype(str).str.contains(pattern, regex=True)
df_filtered = df.loc[mask & df["regular"].notna()].copy()
df_filtered.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"📄 Tổng Alpha ban đầu: {len(df)}")
print(f"✅ Alpha giữ lại: {len(df_filtered)}")
print(f"❌ Alpha bị loại: {len(df) - len(df_filtered)}")
print(f"🚫 Số biến bị loại: {len(REMOVE_VARIABLES)}")
print(f"💾 Đã lưu: {OUTPUT_FILE}")

print("\nDanh sách biến bị loại:")
for variable in REMOVE_VARIABLES:
    print(variable)

In [23]:
import pandas as pd,shutil

FILE="loc_single_0_8_backup.csv";BACKUP="loc_single_0_8_backup_2.csv"

OPTION8=['implied_volatility_call_120','implied_volatility_call_270','implied_volatility_put_270','implied_volatility_put_120','implied_volatility_call_180','implied_volatility_call_60','implied_volatility_put_180','implied_volatility_call_1080','implied_volatility_call_150','implied_volatility_put_60']
MODEL30=['star_new_eps_smart_estimate_fq1','mdl30_smartestimate_fq1_eps','mdl30_numnly_fq1_eps','mdl30_numnly_fy1_eps','mdl30_smartestimate_fq2_eps','star_new_eps_smart_estimate_fq2','star_new_eps_smart_estimate_fy2','star_new_eps_smart_estimate_12m','mdl30_psprise_pct_fq2_eps','mdl30_numnly_fq2_eps']
BAD=OPTION8+MODEL30

shutil.copy2(FILE,BACKUP)
df=pd.read_csv(FILE)

if "regular" not in df.columns:raise ValueError("Không tìm thấy cột regular trong loc_single.csv")

s=df["regular"].fillna("").astype(str).str.lower()
mask_option=s.apply(lambda x:any(v in x for v in OPTION8))
mask_mdl30=s.apply(lambda x:any(v in x for v in MODEL30))
mask=mask_option|mask_mdl30

print("Tổng single ban đầu:",len(df))
print("Single chứa OPTION8:",mask_option.sum())
print("Single chứa MODEL30:",mask_mdl30.sum())
print("Tổng single sẽ xóa:",mask.sum())

df=df.loc[~mask].copy()
df.to_csv(FILE,index=False,encoding="utf-8-sig")

# Check lại sau khi lưu
check=pd.read_csv(FILE)
s2=check["regular"].fillna("").astype(str).str.lower()
left=s2.apply(lambda x:any(v in x for v in BAD)).sum()

print("Single còn lại:",len(df))
print("Còn OPTION8/MODEL30:",left)
print("Đã ghi đè:",FILE)
print("Backup:",BACKUP)

Tổng single ban đầu: 3468
Single chứa OPTION8: 177
Single chứa MODEL30: 198
Tổng single sẽ xóa: 351
Single còn lại: 3117
Còn OPTION8/MODEL30: 0
Đã ghi đè: loc_single_0_8_backup.csv
Backup: loc_single_0_8_backup_2.csv


In [20]:
import pandas as pd,random,re
from collections import Counter

INPUT="loc_single.csv";OUTPUT="alpha_single.txt";OUTPUT_MAP="alpha_single_map.csv";NUM=10000
DATASETS=["NEWS12","ANALYST4","MODEL26","ANALYST15"]
PROFILES=[("A4_M26_NEWS",["ANALYST4","MODEL26","NEWS12"],0.35),("A15_A4_NEWS",["ANALYST15","ANALYST4","NEWS12"],0.25),("A4_NEWS",["ANALYST4","NEWS12"],0.15),("A15_A4_M26_NEWS",["ANALYST15","ANALYST4","MODEL26","NEWS12"],0.20),("A4_M26_A15",["ANALYST4","MODEL26","ANALYST15"],0.05)]
THRESHOLDS=[0.95,0.90];THRESHOLD_WEIGHTS=[0.80,0.20]
N_VALUES=[6,7];N_WEIGHTS=[0.50,0.50]
MAX_OVERLAP=4
MAX_SHARPE=10.0;MAX_FITNESS=10.0

df=pd.read_csv(INPUT)
df["sharpe"]=pd.to_numeric(df["sharpe"],errors="coerce");df["fitness"]=pd.to_numeric(df["fitness"],errors="coerce")
df=df.dropna(subset=["sharpe","regular"]).copy()
df=df[(df["sharpe"]>=0.90)&(df["sharpe"]<=MAX_SHARPE)&(df["fitness"].isna()|((df["fitness"]>=0)&(df["fitness"]<=MAX_FITNESS)))].copy()
df["regular"]=df["regular"].astype(str).str.strip();df=df.drop_duplicates("regular").reset_index(drop=True)

def tags(x):
    x=str(x).lower();a=[]
    if "news_" in x or "nws12_" in x:a.append("NEWS12")
    if "anl4_" in x:a.append("ANALYST4")
    if "mdl26_" in x or "star_arm_country_rank" in x:a.append("MODEL26")
    if "anl15_" in x:a.append("ANALYST15")
    return a

df["tags"]=df["regular"].apply(tags);df=df[df["tags"].apply(lambda x:len(x)>0)].copy()
df["sid"]=range(len(df))

POOLS={}
for t in THRESHOLDS:
    POOLS[t]={d:df[(df["sharpe"]>=t)&df["tags"].apply(lambda x:d in x)].reset_index(drop=True) for d in DATASETS}

print("\nPOOL")
for t in THRESHOLDS:
    print(f"\nSharpe >= {t}")
    for d in DATASETS:print(f"{d:10s}: {len(POOLS[t][d])}")

usage=Counter()

def join_expr(a):
    out=a[0].strip()
    for x in a[1:]:
        x=x.strip()
        if x.startswith("-"):out+=" - "+x[1:].lstrip()
        elif x.startswith("+"):out+=" + "+x[1:].lstrip()
        else:out+=" + "+x
    return out

def choose_profile():
    p=random.choices(PROFILES,weights=[x[2] for x in PROFILES],k=1)[0]
    return p[0],p[1].copy()

def allocation(name,ds,n):
    # Chỉ giữ 15% exploitation 2+2+2, 85% còn lại random
    if name=="A4_M26_NEWS" and n==6 and random.random()<0.15:return {"ANALYST4":2,"MODEL26":2,"NEWS12":2}
    a={d:1 for d in ds}
    for _ in range(n-len(ds)):a[random.choice(ds)]+=1
    return a

def weighted_pick(dataset,count,t,used):
    p=POOLS[t][dataset];p=p[~p["sid"].isin(used)].copy()
    if len(p)<count:return None
    # Single càng bị dùng nhiều thì xác suất được chọn càng giảm
    w=[1/(1+usage[int(s)])**0.75 for s in p["sid"]]
    chosen=[];idx=list(range(len(p)))
    for _ in range(count):
        if not idx:return None
        ww=[w[i] for i in idx];i=random.choices(idx,weights=ww,k=1)[0]
        chosen.append(p.iloc[i].to_dict());idx.remove(i)
    return chosen

def generate():
    t=random.choices(THRESHOLDS,weights=THRESHOLD_WEIGHTS,k=1)[0];n=random.choices(N_VALUES,weights=N_WEIGHTS,k=1)[0]
    name,ds=choose_profile();a=allocation(name,ds,n);selected=[];used=set()
    for d in ds:
        rows=weighted_pick(d,a[d],t,used)
        if rows is None:return None
        for r in rows:
            sid=int(r["sid"])
            if sid in used:continue
            selected.append(r);used.add(sid)
    if len(selected)!=n:return None
    random.shuffle(selected)
    return {"formula":join_expr([x["regular"] for x in selected]),"ids":frozenset(used),"profile":name,"threshold":t,"n":n,"datasets":"+".join(ds),"allocation":"|".join(f"{d}={a[d]}" for d in ds),"min_sharpe":min(x["sharpe"] for x in selected),"mean_sharpe":sum(x["sharpe"] for x in selected)/n,"max_sharpe":max(x["sharpe"] for x in selected)}

def too_similar(ids,accepted_sets):
    # Không nhận nếu chia sẻ > MAX_OVERLAP single với một alpha đã có
    for old in accepted_sets:
        if len(ids&old)>MAX_OVERLAP:return True
    return False

results=[];accepted_sets=[];seen=set();attempts=0;MAX_ATTEMPTS=NUM*300

while len(results)<NUM and attempts<MAX_ATTEMPTS:
    attempts+=1;r=generate()
    if r is None:continue
    key=tuple(sorted(r["ids"]))
    if key in seen:continue
    if too_similar(r["ids"],accepted_sets):continue
    seen.add(key);accepted_sets.append(r["ids"]);results.append(r)
    for sid in r["ids"]:usage[sid]+=1
    # if len(results)%1000==0:print(f"{len(results)}/{NUM} | attempts={attempts}")

random.shuffle(results)

with open(OUTPUT,"w",encoding="utf-8") as f:
    for r in results:f.write(r["formula"]+"\n")

pd.DataFrame([{"row":i+1,"profile":r["profile"],"threshold":r["threshold"],"n_single":r["n"],"datasets":r["datasets"],"allocation":r["allocation"],"min_single_sharpe":r["min_sharpe"],"mean_single_sharpe":r["mean_sharpe"],"max_single_sharpe":r["max_sharpe"],"regular":r["formula"]} for i,r in enumerate(results)]).to_csv(OUTPUT_MAP,index=False,encoding="utf-8-sig")

print("\n"+"="*65)
print("TOTAL:",len(results),"ATTEMPTS:",attempts)
print("TXT:",OUTPUT);print("MAP:",OUTPUT_MAP)
print("\nPROFILE");print(pd.Series([r["profile"] for r in results]).value_counts().to_string())
print("\nTHRESHOLD");print(pd.Series([r["threshold"] for r in results]).value_counts().sort_index(ascending=False).to_string())
print("\nN SINGLE");print(pd.Series([r["n"] for r in results]).value_counts().sort_index().to_string())
print("\nTOP ALLOCATION");print(pd.Series([r["allocation"] for r in results]).value_counts().head(20).to_string())
print("\nSINGLE USAGE");print(pd.Series(list(usage.values())).describe().round(2).to_string())
print("\nDONE")


POOL

Sharpe >= 0.95
NEWS12    : 264
ANALYST4  : 545
MODEL26   : 354
ANALYST15 : 240

Sharpe >= 0.9
NEWS12    : 347
ANALYST4  : 737
MODEL26   : 408
ANALYST15 : 281

TOTAL: 10000 ATTEMPTS: 10000
TXT: alpha_single.txt
MAP: alpha_single_map.csv

PROFILE
A4_M26_NEWS        3491
A15_A4_NEWS        2462
A15_A4_M26_NEWS    2027
A4_NEWS            1542
A4_M26_A15          478

THRESHOLD
0.95    8024
0.90    1976

N SINGLE
6    4998
7    5002

TOP ALLOCATION
ANALYST4=2|MODEL26=2|NEWS12=2                603
ANALYST15=2|ANALYST4=2|NEWS12=2              278
ANALYST4=3|NEWS12=3                          275
ANALYST4=3|MODEL26=2|NEWS12=2                260
ANALYST4=2|MODEL26=3|NEWS12=2                260
ANALYST4=3|NEWS12=4                          259
ANALYST4=2|MODEL26=2|NEWS12=3                251
ANALYST4=4|NEWS12=3                          228
ANALYST4=4|NEWS12=2                          196
ANALYST15=3|ANALYST4=2|NEWS12=2              192
ANALYST15=2|ANALYST4=3|NEWS12=2              186
ANALYS

In [5]:
import pandas as pd
import random
import os
# Tham số
n = 6  # <-- Bạn có thể thay đổi giá trị này
num_formulas = 50000

# Đọc dữ liệu
df = pd.read_csv("loc_single.csv")

# Chia theo sharpe
regular_ge_1 = df[df["sharpe"] >= 0.8]["regular"].tolist()
regular_lt_1 = df[df["sharpe"] < 0.8]["regular"].tolist()

def generate_combination(n):
    p = random.random()
    
    # # Chọn theo xác suất
    # if p <= 1:
    selected = random.sample(regular_ge_1, min(n, len(regular_ge_1)))
    # elif p < 0.36:
    #     selected = random.sample(regular_lt_1, min(n, len(regular_lt_1)))
    # else:
    #     max_ge = min(len(regular_ge_1), n - 2)
    #     count_ge = random.randint(2, max_ge) if max_ge >= 2 else max_ge
    #     count_lt = n - count_ge
    #     if count_ge > 0 and count_lt > 0:
    #         selected = random.sample(regular_ge_1, count_ge) + \
    #                    random.sample(regular_lt_1, min(count_lt, len(regular_lt_1)))
    #     else:
    #         selected = random.sample(regular_ge_1 + regular_lt_1, n)

    # Shuffle để đảm bảo thứ tự ngẫu nhiên
    random.shuffle(selected)

    # # Gán dấu trừ ngẫu nhiên (nếu không phải xác suất 0.25)
    # if random.random() >= 1:
    #     count_minus = random.randint(1, min(2, len(selected)))
    #     minus_indices = random.sample(range(len(selected)), count_minus)
    #     # print("minus_indices : ", minus_indices)
        
    #     for i in minus_indices:
    #         expr = selected[i].lstrip()
    #         if expr.startswith("-"):
    #             # Nếu đã có dấu '-', thêm nữa thành '--' → chuyển thành không có dấu
    #             selected[i] = expr.lstrip("-+ ")
    #         else:
    #             # Nếu chưa có '-', thêm vào
    #             selected[i] = "-" + expr.lstrip("+- ")
    #         # print("selected[i] : ", selected[i])

    # Ghép công thức với logic đúng dấu
    expression = selected[0].strip()
    for expr in selected[1:]:
        expr = expr.strip()
        if expr.startswith("-"):
            # Nếu biểu thức đã có dấu trừ → + -expr
            expression += " + " + expr
        elif expr.startswith("+"):
            expression += " + " + expr[1:]
        else:
            expression += " + " + expr

    # Thay thế các chuỗi '+ -' bằng '-'
    expression = expression.replace("+ -", "- ")

    return expression


# Tạo 10,000 công thức (cho phép trùng lặp nhẹ)
formulas = []
for _ in range(num_formulas):
    n = random.choices([6, 7, 8, 9], weights=[0.25, 0.25, 0.25, 0.25])[0]
    formula = generate_combination(n)
    formulas.append(formula)

# Ghi ra file
with open("alpha_single.txt", "w") as f:
    for formula in formulas:
        f.write(formula + "\n")

print(f"✅ Đã tạo {num_formulas} công thức với {n} regular mỗi công thức.")


✅ Đã tạo 50000 công thức với 8 regular mỗi công thức.


In [ ]:
import pandas as pd

INPUT_FILE = "KQ_GLB_add_Mix_INDUSTRY_TOPDIV3000_18_09_26_QUICK_2.csv"
OUTPUT_FILE = "regular_single.txt"

# ======================================================
# Đọc file
# ======================================================
df = pd.read_csv(INPUT_FILE)

# Ép kiểu số
df["sharpe"] = pd.to_numeric(df["sharpe"], errors="coerce")
df["fitness"] = pd.to_numeric(df["fitness"], errors="coerce")

# ======================================================
# Tự động tìm TẤT CẢ cột kết thúc bằng _result
# ======================================================
result_cols = [
    col for col in df.columns
    if col.endswith("_result")
]

# ======================================================
# Điều kiện chính
# ======================================================
condition = (
    (df["sharpe"] >= 1.58) &
    (df["fitness"] >= 2)
)

# ======================================================
# Tất cả *_result phải PASS hoặc None/NaN
# ======================================================
for col in result_cols:
    condition &= (
        df[col].eq("PASS") |
        df[col].isna()
    )

# ======================================================
# Lọc
# ======================================================
filtered_df = df[condition].copy()

# Bỏ regular rỗng
filtered_df = filtered_df[
    filtered_df["regular"].notna()
].copy()

# ======================================================
# Lưu nguyên regular
# Không có dấu " và không có dấu \
# ======================================================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for regular in filtered_df["regular"]:
        f.write(str(regular).strip() + "\n")

# ======================================================
# Kết quả
# ======================================================
print("\n" + "=" * 60)
print("KẾT QUẢ")
print("=" * 60)

print("Tổng alpha ban đầu :", len(df))
print("Alpha đạt điều kiện:", len(filtered_df))
print("Đã lưu             :", OUTPUT_FILE)

if len(filtered_df) > 0:
    print("\nDanh sách alpha:")
    print(
        filtered_df[
            ["id", "sharpe", "fitness"]
        ].to_string(index=False)
    )

✅ Đã tạo 50000 công thức với 7 regular mỗi công thức.


In [10]:
import pandas as pd
import random

INPUT = "loc_single.csv"
OUTPUT = "alpha_single.txt"

DATASETS = [
    "ANALYST15", "MODEL30", "MODEL26",
    "NEWS12", "ANALYST4", "OPTION8"
]

N_PER_GROUP = 2000
N_SINGLE = [6, 7]

df = pd.read_csv(INPUT)
df["sharpe"] = pd.to_numeric(df["sharpe"], errors="coerce")
df = df[df["sharpe"] >= 0.8].dropna(subset=["regular"]).copy()


def get_dataset(x):
    x = str(x).lower()

    if "anl15_" in x:
        return "ANALYST15"
    if "mdl30_" in x or "star_new_eps_" in x:
        return "MODEL30"
    if "mdl26_" in x or "star_arm_country_rank" in x:
        return "MODEL26"
    if "news_" in x or "nws12_" in x:
        return "NEWS12"
    if "anl4_" in x:
        return "ANALYST4"
    if "implied_volatility_" in x:
        return "OPTION8"

    return None


df["dataset"] = df["regular"].apply(get_dataset)
df = df[df["dataset"].isin(DATASETS)].drop_duplicates("regular")

pools = {
    d: df[df["dataset"] == d]["regular"].tolist()
    for d in DATASETS
}


def join_expr(exprs):
    out = exprs[0].strip()

    for x in exprs[1:]:
        x = x.strip()

        if x.startswith("-"):
            out += " - " + x[1:].lstrip()
        elif x.startswith("+"):
            out += " + " + x[1:].lstrip()
        else:
            out += " + " + x

    return out


def generate(k):
    # Random k dataset khác nhau
    chosen = random.sample(DATASETS, k)

    # Random 6 hoặc 7 single
    n = random.choice(N_SINGLE)

    # Mỗi dataset đã chọn bắt buộc xuất hiện ít nhất 1 lần
    allocation = {d: 1 for d in chosen}

    # Random các slot còn lại
    for _ in range(n - k):
        allocation[random.choice(chosen)] += 1

    exprs = []

    for d, count in allocation.items():
        exprs.extend(
            random.sample(pools[d], count)
        )

    # Random thứ tự single
    random.shuffle(exprs)

    return join_expr(exprs)


# ============================================================
# 2/3/4/5/6 DATASET
# MỖI CASE = 1000 ALPHA
# ============================================================

rows = []

for k in [2, 3, 4, 5, 6]:
    print(f"Generating case {k} datasets...")

    for _ in range(N_PER_GROUP):
        rows.append(generate(k))


# ============================================================
# SHUFFLE TOÀN BỘ 5000 ALPHA
# ============================================================

random.shuffle(rows)


# ============================================================
# XÓA DUPLICATE NẾU CÓ
# ============================================================

rows = list(dict.fromkeys(rows))


# ============================================================
# LƯU TXT - MỖI DÒNG 1 ALPHA
# ============================================================

with open(OUTPUT, "w", encoding="utf-8") as f:
    for formula in rows:
        f.write(formula + "\n")


print("\n==============================")
print(f"✅ Output: {OUTPUT}")
print(f"✅ Tổng Alpha: {len(rows)}")
print(f"✅ 2-6 datasets: {N_PER_GROUP} Alpha/case")
print(f"✅ Mỗi Alpha: random {N_SINGLE}")
print("✅ Đã shuffle toàn bộ")
print("==============================")

Generating case 2 datasets...
Generating case 3 datasets...
Generating case 4 datasets...
Generating case 5 datasets...
Generating case 6 datasets...

✅ Output: alpha_single.txt
✅ Tổng Alpha: 10000
✅ 2-6 datasets: 2000 Alpha/case
✅ Mỗi Alpha: random [6, 7]
✅ Đã shuffle toàn bộ


In [11]:
import pandas as pd
import random
import re
from collections import Counter


# ============================================================
# CONFIG
# ============================================================

INPUT = "loc_single.csv"

OUTPUT_TXT = "alpha_single.txt"
OUTPUT_MAP = "alpha_single_map.csv"

NUM_FORMULAS = 10000

# Core mới
CORE = [
    "ANALYST4",
    "NEWS12",
]

# Random thêm
EXTRA = [
    "MODEL30",
    "MODEL26",
    "ANALYST15",
]

DATASETS = CORE + EXTRA


# ============================================================
# DISTRIBUTION
# ============================================================

# 80% n=6, 20% n=7
N_SINGLE_VALUES = [6, 7]
N_SINGLE_WEIGHTS = [0.80, 0.20]

# Số dataset / Alpha
#
# 2 = ANALYST4 + NEWS12
# 3 = core + 1 extra
# 4 = core + 2 extra
# 5 = core + 3 extra
DATASET_COUNT_VALUES = [2, 3, 4, 5]

DATASET_COUNT_WEIGHTS = [
    0.20,
    0.40,
    0.30,
    0.10,
]

# 70% dùng pool Sharpe >= 0.9
# 30% dùng pool Sharpe >= 0.8
SHARPE_THRESHOLDS = [0.9, 0.8]
SHARPE_WEIGHTS = [0.70, 0.30]

SEED = None

if SEED is not None:
    random.seed(SEED)


# ============================================================
# READ SINGLE
# ============================================================

df = pd.read_csv(INPUT)

for col in ["sharpe", "regular"]:

    if col not in df.columns:
        raise ValueError(f"❌ Thiếu cột: {col}")


df["sharpe"] = pd.to_numeric(
    df["sharpe"],
    errors="coerce"
)

df = df.dropna(
    subset=["sharpe", "regular"]
).copy()

df["regular"] = (
    df["regular"]
    .astype(str)
    .str.strip()
)

# Chỉ cần giữ single >= 0.8
df = df[
    df["sharpe"] >= 0.8
].copy()


# ============================================================
# DETECT DATASET
# ============================================================

def get_dataset(expr):

    x = str(expr).lower()

    # ANALYST15
    if "anl15_" in x:
        return "ANALYST15"

    # MODEL30
    if (
        "mdl30_" in x
        or "star_new_eps_" in x
    ):
        return "MODEL30"

    # MODEL26
    if (
        "mdl26_" in x
        or "star_arm_country_rank" in x
    ):
        return "MODEL26"

    # NEWS12
    if (
        "news_" in x
        or "nws12_" in x
    ):
        return "NEWS12"

    # ANALYST4
    if "anl4_" in x:
        return "ANALYST4"

    # OPTION8 không dùng
    if "implied_volatility_" in x:
        return "OPTION8"

    return None


df["dataset"] = df["regular"].apply(
    get_dataset
)

# Chỉ giữ 5 dataset cần test
df = df[
    df["dataset"].isin(DATASETS)
].copy()


# ============================================================
# NORMALIZE
# ============================================================

def normalize_expr(expr):

    expr = str(expr).strip()

    if expr.startswith("+"):
        expr = expr[1:].strip()

    return re.sub(
        r"\s+",
        " ",
        expr
    )


df["key"] = df["regular"].apply(
    normalize_expr
)

# Xóa duplicate single
df = df.drop_duplicates(
    subset=["key"]
).copy()


# ============================================================
# BUILD POOLS
#
# POOLS[0.9]["ANALYST4"]
# POOLS[0.8]["NEWS12"]
# ...
# ============================================================

POOLS = {}

for threshold in SHARPE_THRESHOLDS:

    POOLS[threshold] = {}

    for dataset in DATASETS:

        pool = df[
            (df["dataset"] == dataset)
            &
            (df["sharpe"] >= threshold)
        ].copy()

        POOLS[threshold][dataset] = (
            pool.reset_index(drop=True)
        )


# ============================================================
# PRINT POOL
# ============================================================

print("\n" + "=" * 65)
print("SINGLE POOLS")
print("=" * 65)

for threshold in SHARPE_THRESHOLDS:

    print(f"\nSharpe >= {threshold}")

    for dataset in DATASETS:

        count = len(
            POOLS[threshold][dataset]
        )

        print(
            f"{dataset:12s}: {count}"
        )

        if count == 0:

            raise ValueError(
                f"❌ Pool rỗng: "
                f"{dataset}, Sharpe >= {threshold}"
            )


# ============================================================
# JOIN EXPRESSIONS
# ============================================================

def join_expr(exprs):

    out = exprs[0].strip()

    for x in exprs[1:]:

        x = x.strip()

        if x.startswith("-"):

            out += (
                " - " +
                x[1:].lstrip()
            )

        elif x.startswith("+"):

            out += (
                " + " +
                x[1:].lstrip()
            )

        else:

            out += (
                " + " +
                x
            )

    return out


# ============================================================
# CHOOSE DATASETS
#
# CORE luôn luôn có:
#
# ANALYST4 + NEWS12
#
# Sau đó random thêm EXTRA.
# ============================================================

def choose_datasets(k):

    extras_needed = (
        k - len(CORE)
    )

    if extras_needed == 0:

        return CORE.copy()

    extras = random.sample(
        EXTRA,
        extras_needed
    )

    return (
        CORE.copy()
        +
        extras
    )


# ============================================================
# ALLOCATION
#
# Mỗi dataset bắt buộc >= 1 single.
#
# Ví dụ:
#
# A4 + NEWS12, n=6
#
# có thể:
# 1+5
# 2+4
# 3+3
# 4+2
# 5+1
#
# Với 3 dataset:
# 4+1+1
# 3+2+1
# 2+2+2
# ...
# ============================================================

def make_allocation(
    datasets,
    n
):

    allocation = {
        d: 1
        for d in datasets
    }

    remaining = (
        n - len(datasets)
    )

    for _ in range(remaining):

        d = random.choice(
            datasets
        )

        allocation[d] += 1

    return allocation


# ============================================================
# PICK SINGLE
#
# Uniform random.
#
# Không weight Fitness.
# Không weight Sharpe bên trong pool.
# ============================================================

def pick_singles(
    dataset,
    count,
    threshold,
    used_keys
):

    pool = POOLS[
        threshold
    ][dataset]

    available = pool[
        ~pool["key"].isin(
            used_keys
        )
    ]

    if len(available) < count:

        return None

    indexes = random.sample(
        range(len(available)),
        count
    )

    return [
        available.iloc[i]
        for i in indexes
    ]


# ============================================================
# GENERATE ONE
# ============================================================

def generate_one():

    # --------------------------------------------------------
    # Sharpe mode
    # --------------------------------------------------------

    threshold = random.choices(
        SHARPE_THRESHOLDS,
        weights=SHARPE_WEIGHTS,
        k=1
    )[0]

    # --------------------------------------------------------
    # 6 / 7 single
    # --------------------------------------------------------

    n = random.choices(
        N_SINGLE_VALUES,
        weights=N_SINGLE_WEIGHTS,
        k=1
    )[0]

    # --------------------------------------------------------
    # 2 / 3 / 4 / 5 dataset
    # --------------------------------------------------------

    k = random.choices(
        DATASET_COUNT_VALUES,
        weights=DATASET_COUNT_WEIGHTS,
        k=1
    )[0]

    # --------------------------------------------------------
    # Dataset combination
    # --------------------------------------------------------

    datasets = choose_datasets(
        k
    )

    # --------------------------------------------------------
    # Allocation
    # --------------------------------------------------------

    allocation = make_allocation(
        datasets,
        n
    )

    selected = []

    used_keys = set()

    # --------------------------------------------------------
    # Pick singles
    # --------------------------------------------------------

    for dataset in datasets:

        count = allocation[
            dataset
        ]

        rows = pick_singles(
            dataset,
            count,
            threshold,
            used_keys
        )

        if rows is None:
            return None

        for row in rows:

            selected.append(
                {
                    "dataset": dataset,
                    "regular": str(
                        row["regular"]
                    ).strip(),
                    "key": row["key"],
                    "sharpe": float(
                        row["sharpe"]
                    ),
                }
            )

            used_keys.add(
                row["key"]
            )

    if len(selected) != n:
        return None

    # --------------------------------------------------------
    # Shuffle single order
    # --------------------------------------------------------

    random.shuffle(
        selected
    )

    expressions = [
        x["regular"]
        for x in selected
    ]

    formula = join_expr(
        expressions
    )

    # Cùng bộ single nhưng thứ tự khác
    # vẫn tính là duplicate.
    combination_key = tuple(
        sorted(
            used_keys
        )
    )

    return {
        "formula": formula,
        "key": combination_key,
        "threshold": threshold,
        "n_single": n,
        "dataset_count": k,
        "datasets": tuple(
            sorted(datasets)
        ),
        "allocation": allocation,
        "selected": selected,
    }


# ============================================================
# GENERATE 10,000 UNIQUE
# ============================================================

results = []

seen = set()

attempts = 0

MAX_ATTEMPTS = (
    NUM_FORMULAS * 100
)


while len(results) < NUM_FORMULAS:

    attempts += 1

    if attempts > MAX_ATTEMPTS:

        print(
            "\n⚠️ Đạt MAX_ATTEMPTS."
        )

        break

    result = generate_one()

    if result is None:
        continue

    if result["key"] in seen:
        continue

    seen.add(
        result["key"]
    )

    results.append(
        result
    )


# ============================================================
# SHUFFLE TOÀN BỘ
#
# TXT và MAP vẫn giữ cùng thứ tự.
# ============================================================

random.shuffle(
    results
)


# ============================================================
# SAVE alpha_single.txt
#
# Mỗi dòng = 1 Alpha.
# ============================================================

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    for result in results:

        f.write(
            result["formula"] + "\n"
        )


# ============================================================
# SAVE MAPPING
#
# Dùng file này để phân tích batch sau.
# ============================================================

mapping = []

for i, result in enumerate(
    results,
    start=1
):

    allocation_text = "|".join(
        f"{d}={result['allocation'][d]}"
        for d in sorted(
            result["allocation"]
        )
    )

    datasets_text = "+".join(
        result["datasets"]
    )

    single_sharpes = [
        x["sharpe"]
        for x in result["selected"]
    ]

    mapping.append(
        {
            "row": i,
            "threshold": result["threshold"],
            "n_single": result["n_single"],
            "dataset_count": result["dataset_count"],
            "datasets": datasets_text,
            "allocation": allocation_text,

            # Sharpe thực tế của 6/7 single
            "min_single_sharpe": min(
                single_sharpes
            ),
            "mean_single_sharpe": sum(
                single_sharpes
            ) / len(single_sharpes),
            "max_single_sharpe": max(
                single_sharpes
            ),

            "regular": result["formula"],
        }
    )


map_df = pd.DataFrame(
    mapping
)

map_df.to_csv(
    OUTPUT_MAP,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# STATS
# ============================================================

threshold_counter = Counter()
n_counter = Counter()
k_counter = Counter()
combo_counter = Counter()
allocation_counter = Counter()

for result in results:

    threshold_counter[
        result["threshold"]
    ] += 1

    n_counter[
        result["n_single"]
    ] += 1

    k_counter[
        result["dataset_count"]
    ] += 1

    combo = "+".join(
        result["datasets"]
    )

    combo_counter[
        combo
    ] += 1

    allocation = "|".join(
        f"{d}={result['allocation'][d]}"
        for d in sorted(
            result["allocation"]
        )
    )

    allocation_counter[
        allocation
    ] += 1


# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("KẾT QUẢ")
print("=" * 70)

print(
    f"Unique Alpha : {len(results)}"
)

print(
    f"Attempts     : {attempts}"
)

print(
    f"TXT          : {OUTPUT_TXT}"
)

print(
    f"Mapping      : {OUTPUT_MAP}"
)


# ============================================================
# SHARPE MODE
# ============================================================

print("\nSHARPE THRESHOLD")

for threshold in sorted(
    threshold_counter,
    reverse=True
):

    count = threshold_counter[
        threshold
    ]

    pct = (
        count /
        len(results) *
        100
    )

    print(
        f">= {threshold}: "
        f"{count:5d} "
        f"({pct:6.2f}%)"
    )


# ============================================================
# N SINGLE
# ============================================================

print("\nN SINGLE")

for n in sorted(
    n_counter
):

    count = n_counter[n]

    pct = (
        count /
        len(results) *
        100
    )

    print(
        f"n={n}: "
        f"{count:5d} "
        f"({pct:6.2f}%)"
    )


# ============================================================
# DATASET COUNT
# ============================================================

print("\nDATASET COUNT")

for k in sorted(
    k_counter
):

    count = k_counter[k]

    pct = (
        count /
        len(results) *
        100
    )

    print(
        f"k={k}: "
        f"{count:5d} "
        f"({pct:6.2f}%)"
    )


# ============================================================
# DATASET COMBINATION
# ============================================================

print("\nDATASET COMBINATIONS")

for combo, count in (
    combo_counter.most_common()
):

    pct = (
        count /
        len(results) *
        100
    )

    print(
        f"{count:5d} "
        f"({pct:6.2f}%) | "
        f"{combo}"
    )


# ============================================================
# TOP ALLOCATIONS
# ============================================================

print("\nTOP 20 ALLOCATIONS")

for allocation, count in (
    allocation_counter
    .most_common(20)
):

    print(
        f"{count:5d} | "
        f"{allocation}"
    )


print("\n" + "=" * 70)
print("✅ DONE")
print("=" * 70)


SINGLE POOLS

Sharpe >= 0.9
ANALYST4    : 170
NEWS12      : 78
MODEL30     : 172
MODEL26     : 153
ANALYST15   : 138

Sharpe >= 0.8
ANALYST4    : 250
NEWS12      : 119
MODEL30     : 253
MODEL26     : 217
ANALYST15   : 208

KẾT QUẢ
Unique Alpha : 10000
Attempts     : 10000
TXT          : alpha_single.txt
Mapping      : alpha_single_map.csv

SHARPE THRESHOLD
>= 0.9:  7008 ( 70.08%)
>= 0.8:  2992 ( 29.92%)

N SINGLE
n=6:  8001 ( 80.01%)
n=7:  1999 ( 19.99%)

DATASET COUNT
k=2:  1915 ( 19.15%)
k=3:  4055 ( 40.55%)
k=4:  3003 ( 30.03%)
k=5:  1027 ( 10.27%)

DATASET COMBINATIONS
 1915 ( 19.15%) | ANALYST4+NEWS12
 1376 ( 13.76%) | ANALYST15+ANALYST4+NEWS12
 1365 ( 13.65%) | ANALYST4+MODEL26+NEWS12
 1314 ( 13.14%) | ANALYST4+MODEL30+NEWS12
 1052 ( 10.52%) | ANALYST4+MODEL26+MODEL30+NEWS12
 1027 ( 10.27%) | ANALYST15+ANALYST4+MODEL26+MODEL30+NEWS12
  977 (  9.77%) | ANALYST15+ANALYST4+MODEL26+NEWS12
  974 (  9.74%) | ANALYST15+ANALYST4+MODEL30+NEWS12

TOP 20 ALLOCATIONS
  591 | ANALYST4=3|NEWS

In [ ]:
import pandas as pd
import random
import re
from collections import Counter


# ============================================================
# CONFIG
# ============================================================

INPUT = "alpha_single.csv"
OUTPUT = "alpha_single.txt"

NUM_FORMULAS = 10000

# Core luôn có
CORE = [
    "ANALYST15",
    "NEWS12",
]

# Dataset được random thêm
EXTRA = [
    "MODEL30",
    "MODEL26",
    "ANALYST4",
]

DATASETS = CORE + EXTRA

# 6 single = 60%
# 7 single = 40%
N_SINGLE_VALUES = [6, 7]
N_SINGLE_WEIGHTS = [0.60, 0.40]

# ------------------------------------------------------------
# Số dataset trong 1 Alpha
#
# Có thêm case riêng:
# ANALYST15 + NEWS12
#
# Phần còn lại ưu tiên 3-4 dataset.
# ------------------------------------------------------------

DATASET_COUNT_VALUES = [2, 3, 4, 5]

DATASET_COUNT_WEIGHTS = [
    0.15,   # 2 dataset: ANALYST15 + NEWS12
    0.35,   # 3 dataset
    0.35,   # 4 dataset
    0.15,   # 5 dataset
]

# ------------------------------------------------------------
# Sharpe mode
#
# 70% Alpha:
# tất cả single được lấy từ pool Sharpe >= 0.9
#
# 30% Alpha:
# tất cả single được lấy từ pool Sharpe >= 0.8
# ------------------------------------------------------------

SHARPE_THRESHOLDS = [0.9, 0.8]
SHARPE_WEIGHTS = [0.70, 0.30]

# None = mỗi lần chạy random khác nhau
SEED = None

if SEED is not None:
    random.seed(SEED)


# ============================================================
# READ SINGLE
# ============================================================

df = pd.read_csv(INPUT)

required = [
    "sharpe",
    "regular",
]

for col in required:

    if col not in df.columns:

        raise ValueError(
            f"❌ Thiếu cột: {col}"
        )


df["sharpe"] = pd.to_numeric(
    df["sharpe"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "sharpe",
        "regular",
    ]
).copy()

df["regular"] = (
    df["regular"]
    .astype(str)
    .str.strip()
)

# Chỉ cần giữ >= 0.8
df = df[
    df["sharpe"] >= 0.8
].copy()


# ============================================================
# DETECT DATASET
# ============================================================

def get_dataset(expr):

    x = str(expr).lower()

    # ANALYST15
    if "anl15_" in x:
        return "ANALYST15"

    # MODEL30
    if (
        "mdl30_" in x
        or
        "star_new_eps_" in x
    ):
        return "MODEL30"

    # MODEL26
    if (
        "mdl26_" in x
        or
        "star_arm_country_rank" in x
    ):
        return "MODEL26"

    # NEWS12
    if (
        "news_" in x
        or
        "nws12_" in x
    ):
        return "NEWS12"

    # ANALYST4
    if "anl4_" in x:
        return "ANALYST4"

    # OPTION8 bỏ hoàn toàn
    if "implied_volatility_" in x:
        return "OPTION8"

    return None


df["dataset"] = df["regular"].apply(
    get_dataset
)


# ============================================================
# CHỈ GIỮ 5 DATASET
# ============================================================

df = df[
    df["dataset"].isin(
        DATASETS
    )
].copy()


# ============================================================
# NORMALIZE KEY
# ============================================================

def normalize_expr(expr):

    expr = str(expr).strip()

    if expr.startswith("+"):
        expr = expr[1:].strip()

    expr = re.sub(
        r"\s+",
        " ",
        expr
    )

    return expr


df["key"] = df["regular"].apply(
    normalize_expr
)

df = df.drop_duplicates(
    subset=["key"]
).copy()


# ============================================================
# TẠO POOL THEO SHARPE
#
# POOLS[0.9]["ANALYST15"]
# POOLS[0.8]["NEWS12"]
# ...
# ============================================================

POOLS = {}

for threshold in SHARPE_THRESHOLDS:

    POOLS[threshold] = {}

    for dataset in DATASETS:

        pool = df[
            (df["dataset"] == dataset)
            &
            (df["sharpe"] >= threshold)
        ].copy()

        POOLS[threshold][dataset] = (
            pool.reset_index(drop=True)
        )


# ============================================================
# PRINT POOL
# ============================================================

print("\n" + "=" * 65)
print("POOL SINGLE")
print("=" * 65)

for threshold in SHARPE_THRESHOLDS:

    print(
        f"\nSharpe >= {threshold}"
    )

    for dataset in DATASETS:

        print(
            f"{dataset:12s}: "
            f"{len(POOLS[threshold][dataset])}"
        )


# ============================================================
# KIỂM TRA POOL
# ============================================================

for threshold in SHARPE_THRESHOLDS:

    for dataset in DATASETS:

        if len(
            POOLS[threshold][dataset]
        ) == 0:

            raise ValueError(
                f"❌ Không có single "
                f"{dataset} với Sharpe >= "
                f"{threshold}"
            )


# ============================================================
# JOIN EXPRESSION
#
# Giữ nguyên dấu của single.
# ============================================================

def join_expr(exprs):

    out = exprs[0].strip()

    for x in exprs[1:]:

        x = x.strip()

        if x.startswith("-"):

            out += (
                " - " +
                x[1:].lstrip()
            )

        elif x.startswith("+"):

            out += (
                " + " +
                x[1:].lstrip()
            )

        else:

            out += (
                " + " +
                x
            )

    return out


# ============================================================
# RANDOM DATASET
#
# ANALYST15 + NEWS12 LUÔN CÓ.
#
# k = 2:
# ANALYST15 + NEWS12
#
# k = 3:
# core + random 1 EXTRA
#
# k = 4:
# core + random 2 EXTRA
#
# k = 5:
# core + cả 3 EXTRA
# ============================================================

def choose_datasets(k):

    if k == 2:

        return CORE.copy()

    extras_needed = (
        k - len(CORE)
    )

    extras = random.sample(
        EXTRA,
        extras_needed
    )

    return (
        CORE.copy()
        +
        extras
    )


# ============================================================
# ALLOCATION
#
# Mỗi dataset đã chọn bắt buộc >= 1 single.
#
# Ví dụ:
#
# ANALYST15 + NEWS12
# n = 6
#
# có thể:
# 3 + 3
# 4 + 2
# 2 + 4
# 5 + 1
# 1 + 5
#
#
# Với 3 dataset:
#
# 2+2+2
# 3+2+1
# 4+1+1
# ...
# ============================================================

def make_allocation(
    datasets,
    n
):

    allocation = {
        dataset: 1
        for dataset in datasets
    }

    remaining = (
        n -
        len(datasets)
    )

    for _ in range(
        remaining
    ):

        dataset = random.choice(
            datasets
        )

        allocation[
            dataset
        ] += 1

    return allocation


# ============================================================
# PICK SINGLE
# ============================================================

def pick_singles(
    dataset,
    count,
    threshold,
    used_keys
):

    pool = (
        POOLS[threshold][dataset]
    )

    # Chỉ lấy những single chưa dùng
    available = pool[
        ~pool["key"].isin(
            used_keys
        )
    ]

    if len(available) < count:

        return None

    # Random uniform
    indexes = random.sample(
        range(len(available)),
        count
    )

    rows = [
        available.iloc[i]
        for i in indexes
    ]

    return rows


# ============================================================
# GENERATE 1 ALPHA
# ============================================================

def generate_one():

    # --------------------------------------------------------
    # 70% Sharpe >= 0.9
    # 30% Sharpe >= 0.8
    # --------------------------------------------------------

    threshold = random.choices(
        SHARPE_THRESHOLDS,
        weights=SHARPE_WEIGHTS,
        k=1
    )[0]

    # --------------------------------------------------------
    # Random 6 / 7 single
    # --------------------------------------------------------

    n = random.choices(
        N_SINGLE_VALUES,
        weights=N_SINGLE_WEIGHTS,
        k=1
    )[0]

    # --------------------------------------------------------
    # Random 2 / 3 / 4 / 5 dataset
    # --------------------------------------------------------

    k = random.choices(
        DATASET_COUNT_VALUES,
        weights=DATASET_COUNT_WEIGHTS,
        k=1
    )[0]

    # --------------------------------------------------------
    # Chọn dataset
    # --------------------------------------------------------

    datasets = choose_datasets(
        k
    )

    # --------------------------------------------------------
    # Allocation
    # --------------------------------------------------------

    allocation = make_allocation(
        datasets,
        n
    )

    selected = []

    used_keys = set()

    # --------------------------------------------------------
    # Chọn single
    # --------------------------------------------------------

    for dataset in datasets:

        count = allocation[
            dataset
        ]

        rows = pick_singles(
            dataset,
            count,
            threshold,
            used_keys
        )

        if rows is None:

            return None

        for row in rows:

            selected.append(
                str(
                    row["regular"]
                ).strip()
            )

            used_keys.add(
                row["key"]
            )

    # --------------------------------------------------------
    # Safety
    # --------------------------------------------------------

    if len(selected) != n:

        return None

    # --------------------------------------------------------
    # Shuffle thứ tự single
    # --------------------------------------------------------

    random.shuffle(
        selected
    )

    formula = join_expr(
        selected
    )

    # --------------------------------------------------------
    # Duplicate key
    #
    # Cùng các single nhưng khác thứ tự
    # vẫn xem là cùng combination.
    # --------------------------------------------------------

    combination_key = tuple(
        sorted(
            used_keys
        )
    )

    return {
        "formula": formula,
        "key": combination_key,
        "threshold": threshold,
        "n": n,
        "k": k,
        "datasets": tuple(
            sorted(datasets)
        ),
        "allocation": allocation,
    }


# ============================================================
# GENERATE 10,000 UNIQUE
# ============================================================

formulas = []

seen = set()

sharpe_counter = Counter()
n_counter = Counter()
k_counter = Counter()
dataset_combo_counter = Counter()
allocation_counter = Counter()

attempts = 0

MAX_ATTEMPTS = (
    NUM_FORMULAS * 100
)


while (
    len(formulas) <
    NUM_FORMULAS
):

    attempts += 1

    if attempts > MAX_ATTEMPTS:

        print(
            "\n⚠️ Đạt MAX_ATTEMPTS."
        )

        break

    result = generate_one()

    if result is None:

        continue

    if result["key"] in seen:

        continue

    seen.add(
        result["key"]
    )

    formulas.append(
        result["formula"]
    )

    # --------------------------------------------------------
    # STATS
    # --------------------------------------------------------

    sharpe_counter[
        result["threshold"]
    ] += 1

    n_counter[
        result["n"]
    ] += 1

    k_counter[
        result["k"]
    ] += 1

    combo_text = "+".join(
        result["datasets"]
    )

    dataset_combo_counter[
        combo_text
    ] += 1

    allocation_text = "|".join(
        f"{dataset}="
        f"{result['allocation'][dataset]}"
        for dataset in sorted(
            result["allocation"]
        )
    )

    allocation_counter[
        allocation_text
    ] += 1


# ============================================================
# SHUFFLE TOÀN BỘ
# ============================================================

random.shuffle(
    formulas
)


# ============================================================
# SAVE alpha_single.txt
# ============================================================

with open(
    OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    for formula in formulas:

        f.write(
            formula + "\n"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("KẾT QUẢ")
print("=" * 65)

print(
    f"Total unique : {len(formulas)}"
)

print(
    f"Attempts     : {attempts}"
)

print(
    f"Output       : {OUTPUT}"
)


# ============================================================
# SHARPE MODE
# ============================================================

print("\nSHARPE SINGLE")

for threshold in sorted(
    sharpe_counter,
    reverse=True
):

    count = sharpe_counter[
        threshold
    ]

    pct = (
        count /
        len(formulas) *
        100
    )

    print(
        f">= {threshold}: "
        f"{count} "
        f"({pct:.2f}%)"
    )


# ============================================================
# N SINGLE
# ============================================================

print("\nN SINGLE")

for n in sorted(
    n_counter
):

    count = n_counter[n]

    pct = (
        count /
        len(formulas) *
        100
    )

    print(
        f"n={n}: "
        f"{count} "
        f"({pct:.2f}%)"
    )


# ============================================================
# DATASET COUNT
# ============================================================

print("\nDATASET COUNT")

for k in sorted(
    k_counter
):

    count = k_counter[k]

    pct = (
        count /
        len(formulas) *
        100
    )

    print(
        f"k={k}: "
        f"{count} "
        f"({pct:.2f}%)"
    )


# ============================================================
# DATASET COMBINATIONS
# ============================================================

print("\nDATASET COMBINATIONS")

for combo, count in (
    dataset_combo_counter
    .most_common()
):

    pct = (
        count /
        len(formulas) *
        100
    )

    print(
        f"{count:5d} "
        f"({pct:6.2f}%) "
        f"{combo}"
    )


# ============================================================
# TOP ALLOCATION
# ============================================================

print("\nTOP 20 ALLOCATION")

for allocation, count in (
    allocation_counter
    .most_common(20)
):

    print(
        f"{count:5d} | "
        f"{allocation}"
    )


print("\n" + "=" * 65)
print("✅ DONE")
print("=" * 65)

FileNotFoundError: [Errno 2] No such file or directory: 'single.csv'